# NOVA RETAIL — Unified Risk Matrix

## Consolidación Forense de Riesgo por Tienda
**Proyecto:** Diagnóstico de Prevención de Pérdidas  
**Autor:** Denz One  
**Fase:** Integración de señales operativas y económicas en una matriz priorizada  
**Objetivo:** Unificar los principales hallazgos del caso en una sola tabla de riesgo por tienda para priorizar intervención, escalamiento e investigación focalizada.

---

### Propósito de este notebook
Este notebook consolida en una sola vista analítica las señales más relevantes identificadas en fases anteriores:

- severidad de discrepancia entre despacho y recepción
- recepciones en horario atípico (05:00)
- exposición económica a Ghost SKUs
- concentración por ruta logística
- línea de mando regional
- opacidad estructural (`ghost_store`)

### Preguntas clave
1. ¿Qué tiendas combinan múltiples capas de riesgo?
2. ¿Qué señales pesan más en la priorización?
3. ¿Qué tan concentrado está el riesgo económico y operativo?
4. ¿Cómo traducimos hallazgos dispersos en una decisión ejecutiva única?

### Nota del analista
La matriz de riesgo no busca declarar culpabilidad.  
Busca priorizar de forma objetiva los nodos donde convergen vulnerabilidad estructural, pérdida material y comportamiento operativo atípico.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

DATA_PATH = Path("../data/raw")

stores = pd.read_csv(DATA_PATH / "stores.csv")
products_sap = pd.read_csv(DATA_PATH / "products_sap.csv")
dispatches = pd.read_csv(DATA_PATH / "dispatches.csv")
receptions = pd.read_csv(DATA_PATH / "receptions.csv")
products_as400 = pd.read_csv(DATA_PATH / "products_as400.csv")

print("Datasets cargados correctamente.")

Datasets cargados correctamente.


In [2]:
high_value_categories = ["ELECTRONICA", "TELEFONIA", "COMPUTO", "ELECTRODOMESTICOS"]

products_priority = products_sap[
    products_sap["category_sap"].isin(high_value_categories)
].copy()

products_priority = products_priority[
    products_priority["price_sap"] >= products_priority["price_sap"].quantile(0.95)
].copy()

print("SKUs prioritarios:", len(products_priority))

SKUs prioritarios: 315


In [3]:
dispatches_priority = dispatches.merge(
    products_priority[["sku_sap", "category_sap", "price_sap"]],
    left_on="sku",
    right_on="sku_sap",
    how="inner"
)

receptions_priority = receptions.merge(
    products_priority[["sku_sap", "category_sap", "price_sap"]],
    left_on="sku",
    right_on="sku_sap",
    how="inner"
)

merged_priority = dispatches_priority.merge(
    receptions_priority,
    left_on="dispatch_id",
    right_on="dispatch_reference",
    how="inner",
    suffixes=("_dispatch", "_reception")
)

print("Merged priority:", merged_priority.shape)

Merged priority: (5989, 28)


In [4]:
merged_priority["reception_hour"] = (
    pd.to_datetime(merged_priority["reception_time"], format="%H:%M", errors="coerce")
    .dt.hour
)

merged_priority_units = merged_priority[
    merged_priority["unit_type"] == "UNIDAD"
].copy()

print("Merged priority units:", merged_priority_units.shape)

Merged priority units: (4505, 29)


In [5]:
severity_by_store = (
    merged_priority_units
    .groupby("store_id_dispatch")[["quantity_dispatched", "quantity_received"]]
    .sum()
    .reset_index()
)

severity_by_store["quantity_diff"] = (
    severity_by_store["quantity_dispatched"] - severity_by_store["quantity_received"]
)

severity_by_store["pct_diff"] = np.where(
    severity_by_store["quantity_dispatched"] > 0,
    (severity_by_store["quantity_diff"] / severity_by_store["quantity_dispatched"]) * 100,
    np.nan
)

severity_by_store = severity_by_store.sort_values("pct_diff", ascending=False)

severity_by_store.head(15)

,store_id_dispatch,quantity_dispatched,quantity_received,quantity_diff,pct_diff
155,156,122,97,25,20.491803
46,47,148,118,30,20.270270
14,15,156,125,31,19.871795
82,83,83,69,14,16.867470
28,29,142,119,23,16.197183
133,134,136,115,21,15.441176
66,67,162,138,24,14.814815
90,91,170,145,25,14.705882
111,112,174,151,23,13.218391
170,171,206,179,27,13.106796


In [6]:
early_hour_counts = (
    merged_priority_units[merged_priority_units["reception_hour"] == 5]
    .groupby("store_id_dispatch")
    .size()
    .reset_index(name="recepciones_5am")
)

early_hour_counts.sort_values("recepciones_5am", ascending=False).head(15)

,store_id_dispatch,recepciones_5am
0,15,21
9,171,21
2,47,20
3,67,20
8,156,18
1,29,17
5,91,17
7,134,16
6,112,15
4,83,10


In [7]:
apple_ghosts = products_as400[products_as400["is_ghost"] == True].copy()
ghost_sap_refs = apple_ghosts["sku_sap_reference"].dropna().tolist()

ghost_dispatches = dispatches[
    dispatches["sku"].isin(ghost_sap_refs)
].copy()

ghost_dispatch_value = ghost_dispatches.merge(
    products_sap[["sku_sap", "price_sap", "description_sap", "brand", "category_sap"]],
    left_on="sku",
    right_on="sku_sap",
    how="left"
)

ghost_dispatch_value["dispatch_value_mxn"] = (
    ghost_dispatch_value["quantity_dispatched"] * ghost_dispatch_value["price_sap"]
)

ghost_by_store = ghost_dispatch_value.groupby("store_id").agg(
    despachos_ghost=("dispatch_id", "count"),
    unidades_ghost=("quantity_dispatched", "sum"),
    valor_ghost_mxn=("dispatch_value_mxn", "sum")
).reset_index()

ghost_by_store.sort_values("valor_ghost_mxn", ascending=False).head(15)

,store_id,despachos_ghost,unidades_ghost,valor_ghost_mxn
31,32,7,57,1039910.41
103,116,9,59,1020478.72
61,68,9,62,1006412.77
139,155,4,42,776975.70
83,94,5,34,748372.99
57,64,3,26,714577.75
7,8,6,35,699276.93
112,126,4,26,689372.51
2,3,6,37,678703.89
69,77,3,25,658968.83


In [8]:
store_meta = stores[[
    "store_id",
    "store_name",
    "city",
    "state",
    "system",
    "cedis_route",
    "regional_manager_id",
    "ghost_store"
]].copy()

store_meta.head()

,store_id,store_name,city,state,system,cedis_route,regional_manager_id,ghost_store
0,1,NOVA-ZAC-001,Zacatecas,Zacatecas,SAP,SUR,GR-002,False
1,2,NOVA-SAL-002,Salamanca,Guanajuato,SAP,SUR,GR-003,False
2,3,NOVA-APO-003,Apodaca,Nuevo León,SAP,GOLFO,GR-004,False
3,4,NOVA-OBR-004,Obregón,Sonora,SAP,PACIFICO,GR-005,False
4,5,NOVA-SAL-005,Saltillo,Coahuila,SAP,PACIFICO,GR-001,False


In [9]:
risk_matrix = store_meta.merge(
    severity_by_store,
    left_on="store_id",
    right_on="store_id_dispatch",
    how="left"
).merge(
    early_hour_counts,
    left_on="store_id",
    right_on="store_id_dispatch",
    how="left"
).merge(
    ghost_by_store,
    left_on="store_id",
    right_on="store_id",
    how="left"
)

risk_matrix = risk_matrix.drop(columns=[
    "store_id_dispatch_x",
    "store_id_dispatch_y"
], errors="ignore")

risk_matrix["recepciones_5am"] = risk_matrix["recepciones_5am"].fillna(0)
risk_matrix["despachos_ghost"] = risk_matrix["despachos_ghost"].fillna(0)
risk_matrix["unidades_ghost"] = risk_matrix["unidades_ghost"].fillna(0)
risk_matrix["valor_ghost_mxn"] = risk_matrix["valor_ghost_mxn"].fillna(0)
risk_matrix["quantity_dispatched"] = risk_matrix["quantity_dispatched"].fillna(0)
risk_matrix["quantity_received"] = risk_matrix["quantity_received"].fillna(0)
risk_matrix["quantity_diff"] = risk_matrix["quantity_diff"].fillna(0)
risk_matrix["pct_diff"] = risk_matrix["pct_diff"].fillna(0)

risk_matrix.head(20)

,store_id,store_name,city,state,system,cedis_route,regional_manager_id,ghost_store,quantity_dispatched,quantity_received,quantity_diff,pct_diff,recepciones_5am,despachos_ghost,unidades_ghost,valor_ghost_mxn
0,1,NOVA-ZAC-001,Zacatecas,Zacatecas,SAP,SUR,GR-002,False,158,158,0,0.000000,0.0,2.0,18.0,497440.92
1,2,NOVA-SAL-002,Salamanca,Guanajuato,SAP,SUR,GR-003,False,178,178,0,0.000000,0.0,3.0,27.0,443676.42
2,3,NOVA-APO-003,Apodaca,Nuevo León,SAP,GOLFO,GR-004,False,145,145,0,0.000000,0.0,6.0,37.0,678703.89
3,4,NOVA-OBR-004,Obregón,Sonora,SAP,PACIFICO,GR-005,False,95,95,0,0.000000,0.0,1.0,2.0,11445.60
4,5,NOVA-SAL-005,Saltillo,Coahuila,SAP,PACIFICO,GR-001,False,129,127,2,1.550388,0.0,5.0,29.0,510366.95
5,6,NOVA-OBR-006,Obregón,Sonora,SAP,SUR,GR-002,False,155,154,1,0.645161,0.0,2.0,12.0,78227.20
6,7,NOVA-TAM-007,Tampico,Tamaulipas,SAP,SUR,GR-003,False,184,184,0,0.000000,0.0,1.0,12.0,111196.92
7,8,NOVA-CUL-008,Culiacán,Sinaloa,SAP,GOLFO,GR-004,False,187,186,1,0.534759,0.0,6.0,35.0,699276.93
8,9,NOVA-SOL-009,Soledad,San Luis Potosí,SAP,CENTRO,GR-005,False,174,174,0,0.000000,0.0,3.0,11.0,235943.47
9,10,NOVA-MAT-010,Matamoros,Tamaulipas,SAP,CENTRO,GR-001,False,124,123,1,0.806452,0.0,1.0,12.0,68673.60


In [10]:
print("Tiendas en la matriz:", len(risk_matrix))
print("Tiendas con severidad > 0:", (risk_matrix["pct_diff"] > 0).sum())
print("Tiendas con recepciones 5am:", (risk_matrix["recepciones_5am"] > 0).sum())
print("Tiendas con exposición ghost:", (risk_matrix["valor_ghost_mxn"] > 0).sum())

Tiendas en la matriz: 187
Tiendas con severidad > 0: 79
Tiendas con recepciones 5am: 10
Tiendas con exposición ghost: 170


In [11]:
risk_matrix["route_north_flag"] = np.where(risk_matrix["cedis_route"] == "NORTE", 1, 0)
risk_matrix["ghost_store_flag"] = np.where(risk_matrix["ghost_store"] == True, 1, 0)

risk_matrix["pct_diff_norm"] = risk_matrix["pct_diff"] / risk_matrix["pct_diff"].max()
risk_matrix["recepciones_5am_norm"] = risk_matrix["recepciones_5am"] / risk_matrix["recepciones_5am"].max()
risk_matrix["valor_ghost_norm"] = risk_matrix["valor_ghost_mxn"] / risk_matrix["valor_ghost_mxn"].max()

risk_matrix[[
    "store_id",
    "pct_diff",
    "pct_diff_norm",
    "recepciones_5am",
    "recepciones_5am_norm",
    "valor_ghost_mxn",
    "valor_ghost_norm",
    "route_north_flag",
    "ghost_store_flag"
]].head(20)

,store_id,pct_diff,pct_diff_norm,recepciones_5am,recepciones_5am_norm,valor_ghost_mxn,valor_ghost_norm,route_north_flag,ghost_store_flag
0,1,0.000000,0.000000,0.0,0.0,497440.92,0.478350,0,0
1,2,0.000000,0.000000,0.0,0.0,443676.42,0.426649,0,0
2,3,0.000000,0.000000,0.0,0.0,678703.89,0.652656,0,0
3,4,0.000000,0.000000,0.0,0.0,11445.60,0.011006,0,0
4,5,1.550388,0.075659,0.0,0.0,510366.95,0.490780,0,0
5,6,0.645161,0.031484,0.0,0.0,78227.20,0.075225,0,0
6,7,0.000000,0.000000,0.0,0.0,111196.92,0.106929,0,0
7,8,0.534759,0.026096,0.0,0.0,699276.93,0.672440,0,0
8,9,0.000000,0.000000,0.0,0.0,235943.47,0.226888,0,0
9,10,0.806452,0.039355,0.0,0.0,68673.60,0.066038,0,0


In [12]:
risk_matrix["route_north_flag"] = np.where(risk_matrix["cedis_route"] == "NORTE", 1, 0)
risk_matrix["ghost_store_flag"] = np.where(risk_matrix["ghost_store"] == True, 1, 0)

risk_matrix["pct_diff_norm"] = risk_matrix["pct_diff"] / risk_matrix["pct_diff"].max()
risk_matrix["recepciones_5am_norm"] = risk_matrix["recepciones_5am"] / risk_matrix["recepciones_5am"].max()
risk_matrix["valor_ghost_norm"] = risk_matrix["valor_ghost_mxn"] / risk_matrix["valor_ghost_mxn"].max()

risk_matrix[[
    "store_id",
    "pct_diff",
    "pct_diff_norm",
    "recepciones_5am",
    "recepciones_5am_norm",
    "valor_ghost_mxn",
    "valor_ghost_norm",
    "route_north_flag",
    "ghost_store_flag"
]].head(20)

,store_id,pct_diff,pct_diff_norm,recepciones_5am,recepciones_5am_norm,valor_ghost_mxn,valor_ghost_norm,route_north_flag,ghost_store_flag
0,1,0.000000,0.000000,0.0,0.0,497440.92,0.478350,0,0
1,2,0.000000,0.000000,0.0,0.0,443676.42,0.426649,0,0
2,3,0.000000,0.000000,0.0,0.0,678703.89,0.652656,0,0
3,4,0.000000,0.000000,0.0,0.0,11445.60,0.011006,0,0
4,5,1.550388,0.075659,0.0,0.0,510366.95,0.490780,0,0
5,6,0.645161,0.031484,0.0,0.0,78227.20,0.075225,0,0
6,7,0.000000,0.000000,0.0,0.0,111196.92,0.106929,0,0
7,8,0.534759,0.026096,0.0,0.0,699276.93,0.672440,0,0
8,9,0.000000,0.000000,0.0,0.0,235943.47,0.226888,0,0
9,10,0.806452,0.039355,0.0,0.0,68673.60,0.066038,0,0


In [13]:
risk_matrix["risk_score"] = (
    risk_matrix["pct_diff_norm"] * 0.35 +
    risk_matrix["recepciones_5am_norm"] * 0.25 +
    risk_matrix["valor_ghost_norm"] * 0.20 +
    risk_matrix["route_north_flag"] * 0.15 +
    risk_matrix["ghost_store_flag"] * 0.05
)

risk_matrix = risk_matrix.sort_values("risk_score", ascending=False)

risk_matrix[[
    "store_id",
    "store_name",
    "cedis_route",
    "regional_manager_id",
    "pct_diff",
    "recepciones_5am",
    "valor_ghost_mxn",
    "ghost_store",
    "risk_score"
]].head(20)

,store_id,store_name,cedis_route,regional_manager_id,pct_diff,recepciones_5am,valor_ghost_mxn,ghost_store,risk_score
14,15,NOVA-SAN-015,NORTE,GR-001,19.871795,21.0,458997.22,False,0.827687
155,156,NOVA-OBR-156,NORTE,GR-002,20.491803,18.0,556059.02,False,0.821229
46,47,NOVA-MAZ-047,NORTE,GR-002,20.270270,20.0,178316.10,False,0.768606
170,171,NOVA-MAZ-171,NORTE,GR-002,13.106796,21.0,199795.44,True,0.712290
66,67,NOVA-TAM-067,NORTE,GR-003,14.814815,20.0,305484.84,False,0.699884
90,91,NOVA-SAL-091,NORTE,GR-002,14.705882,17.0,308406.34,False,0.662871
133,134,NOVA-CUL-134,NORTE,GR-005,15.441176,16.0,224295.52,False,0.647349
28,29,NOVA-SAN-029,NORTE,GR-005,16.197183,17.0,58487.21,False,0.640277
82,83,NOVA-MEX-083,NORTE,GR-002,16.867470,10.0,302000.45,False,0.615226
111,112,NOVA-OBR-112,NORTE,GR-002,13.218391,15.0,294833.85,False,0.611045


In [14]:
risk_matrix["risk_score_v2"] = (
    risk_matrix["pct_diff_norm"] * 0.40 +
    risk_matrix["recepciones_5am_norm"] * 0.27 +
    risk_matrix["valor_ghost_norm"] * 0.20 +
    risk_matrix["route_north_flag"] * 0.08 +
    risk_matrix["ghost_store_flag"] * 0.05
)

risk_matrix = risk_matrix.sort_values("risk_score_v2", ascending=False)

risk_matrix[[
    "store_id",
    "store_name",
    "cedis_route",
    "regional_manager_id",
    "pct_diff",
    "recepciones_5am",
    "valor_ghost_mxn",
    "ghost_store",
    "risk_score",
    "risk_score_v2"
]].head(20)

,store_id,store_name,cedis_route,regional_manager_id,pct_diff,recepciones_5am,valor_ghost_mxn,ghost_store,risk_score,risk_score_v2
14,15,NOVA-SAN-015,NORTE,GR-001,19.871795,21.0,458997.22,False,0.827687,0.826174
155,156,NOVA-OBR-156,NORTE,GR-002,20.491803,18.0,556059.02,False,0.821229,0.818372
46,47,NOVA-MAZ-047,NORTE,GR-002,20.270270,20.0,178316.10,False,0.768606,0.767113
170,171,NOVA-MAZ-171,NORTE,GR-002,13.106796,21.0,199795.44,True,0.712290,0.694270
66,67,NOVA-TAM-067,NORTE,GR-003,14.814815,20.0,305484.84,False,0.699884,0.685080
90,91,NOVA-SAL-091,NORTE,GR-002,14.705882,17.0,308406.34,False,0.662871,0.644944
133,134,NOVA-CUL-134,NORTE,GR-005,15.441176,16.0,224295.52,False,0.647349,0.630264
28,29,NOVA-SAN-029,NORTE,GR-005,16.197183,17.0,58487.21,False,0.640277,0.625989
82,83,NOVA-MEX-083,NORTE,GR-002,16.867470,10.0,302000.45,False,0.615226,0.595906
111,112,NOVA-OBR-112,NORTE,GR-002,13.218391,15.0,294833.85,False,0.611045,0.587584


In [15]:
def assign_risk_tier(score):
    if score >= 0.60:
        return "TIER 1 - CRITICO"
    elif score >= 0.30:
        return "TIER 2 - ALTO"
    elif score > 0:
        return "TIER 3 - MONITOREO"
    else:
        return "TIER 4 - BASELINE"

risk_matrix["risk_tier"] = risk_matrix["risk_score_v2"].apply(assign_risk_tier)

risk_matrix[[
    "store_id",
    "store_name",
    "cedis_route",
    "pct_diff",
    "recepciones_5am",
    "valor_ghost_mxn",
    "ghost_store",
    "risk_score_v2",
    "risk_tier"
]].head(25)

,store_id,store_name,cedis_route,pct_diff,recepciones_5am,valor_ghost_mxn,ghost_store,risk_score_v2,risk_tier
14,15,NOVA-SAN-015,NORTE,19.871795,21.0,458997.22,False,0.826174,TIER 1 - CRITICO
155,156,NOVA-OBR-156,NORTE,20.491803,18.0,556059.02,False,0.818372,TIER 1 - CRITICO
46,47,NOVA-MAZ-047,NORTE,20.270270,20.0,178316.10,False,0.767113,TIER 1 - CRITICO
170,171,NOVA-MAZ-171,NORTE,13.106796,21.0,199795.44,True,0.694270,TIER 1 - CRITICO
66,67,NOVA-TAM-067,NORTE,14.814815,20.0,305484.84,False,0.685080,TIER 1 - CRITICO
...,...,...,...,...,...,...,...,...,...
98,99,NOVA-DEL-099,PACIFICO,0.751880,0.0,653340.72,False,0.140330,TIER 3 - MONITOREO
63,64,NOVA-SAN-064,PACIFICO,0.000000,0.0,714577.75,False,0.137431,TIER 3 - MONITOREO
125,126,NOVA-MON-126,GOLFO,0.000000,0.0,689372.51,False,0.132583,TIER 3 - MONITOREO
2,3,NOVA-APO-003,GOLFO,0.000000,0.0,678703.89,False,0.130531,TIER 3 - MONITOREO


In [16]:
risk_matrix["risk_tier"].value_counts()

risk_tier
TIER 3 - MONITOREO    166
TIER 4 - BASELINE      11
TIER 1 - CRITICO        8
TIER 2 - ALTO           2
Name: count, dtype: int64

In [17]:
def assign_risk_tier_v2(score):
    if score >= 0.65:
        return "TIER 1 - CRITICO"
    elif score >= 0.40:
        return "TIER 2 - ALTO"
    elif score >= 0.20:
        return "TIER 3 - MONITOREO"
    else:
        return "TIER 4 - BASELINE"

risk_matrix["risk_tier_v2"] = risk_matrix["risk_score_v2"].apply(assign_risk_tier_v2)

risk_matrix["risk_tier_v2"].value_counts()

risk_tier_v2
TIER 4 - BASELINE     176
TIER 1 - CRITICO        5
TIER 2 - ALTO           5
TIER 3 - MONITOREO      1
Name: count, dtype: int64

In [18]:
risk_matrix[[
    "store_id",
    "store_name",
    "cedis_route",
    "pct_diff",
    "recepciones_5am",
    "valor_ghost_mxn",
    "ghost_store",
    "risk_score_v2",
    "risk_tier_v2"
]].head(30)

,store_id,store_name,cedis_route,pct_diff,recepciones_5am,valor_ghost_mxn,ghost_store,risk_score_v2,risk_tier_v2
14,15,NOVA-SAN-015,NORTE,19.871795,21.0,458997.22,False,0.826174,TIER 1 - CRITICO
155,156,NOVA-OBR-156,NORTE,20.491803,18.0,556059.02,False,0.818372,TIER 1 - CRITICO
46,47,NOVA-MAZ-047,NORTE,20.270270,20.0,178316.10,False,0.767113,TIER 1 - CRITICO
170,171,NOVA-MAZ-171,NORTE,13.106796,21.0,199795.44,True,0.694270,TIER 1 - CRITICO
66,67,NOVA-TAM-067,NORTE,14.814815,20.0,305484.84,False,0.685080,TIER 1 - CRITICO
...,...,...,...,...,...,...,...,...,...
76,77,NOVA-LOS-077,GOLFO,0.000000,0.0,658968.83,False,0.126736,TIER 4 - BASELINE
34,35,NOVA-SAL-035,GOLFO,0.396825,0.0,611712.96,False,0.125393,TIER 4 - BASELINE
141,142,NOVA-DEL-142,PACIFICO,0.000000,0.0,650379.39,False,0.125084,TIER 4 - BASELINE
64,65,NOVA-ZAC-065,PACIFICO,0.581395,0.0,590400.95,False,0.124897,TIER 4 - BASELINE


In [19]:
resumen_matriz_riesgo = pd.DataFrame({
    "indicador": [
        "Tiendas totales evaluadas",
        "Tiendas con score > 0",
        "Tiendas Tier 1 - Crítico",
        "Tiendas Tier 2 - Alto",
        "Tiendas Tier 3 - Monitoreo",
        "Tiendas Tier 4 - Baseline",
        "Ruta dominante en Tier 1",
        "Tiendas Tier 1 con recepciones 5 AM"
    ],
    "valor": [
        len(risk_matrix),
        int((risk_matrix["risk_score_v2"] > 0).sum()),
        int((risk_matrix["risk_tier_v2"] == "TIER 1 - CRITICO").sum()),
        int((risk_matrix["risk_tier_v2"] == "TIER 2 - ALTO").sum()),
        int((risk_matrix["risk_tier_v2"] == "TIER 3 - MONITOREO").sum()),
        int((risk_matrix["risk_tier_v2"] == "TIER 4 - BASELINE").sum()),
        risk_matrix[risk_matrix["risk_tier_v2"] == "TIER 1 - CRITICO"]["cedis_route"].mode()[0],
        int(((risk_matrix["risk_tier_v2"] == "TIER 1 - CRITICO") & (risk_matrix["recepciones_5am"] > 0)).sum())
    ]
})

resumen_matriz_riesgo

,indicador,valor
0,Tiendas totales evaluadas,187
1,Tiendas con score > 0,176
2,Tiendas Tier 1 - Crítico,5
3,Tiendas Tier 2 - Alto,5
4,Tiendas Tier 3 - Monitoreo,1
5,Tiendas Tier 4 - Baseline,176
6,Ruta dominante en Tier 1,NORTE
7,Tiendas Tier 1 con recepciones 5 AM,5


## Conclusiones de la matriz de riesgo unificada

### Hallazgos principales
- La matriz consolidó en una sola vista las principales señales del caso: severidad de discrepancia, recepciones 5 AM, exposición a Ghost SKUs, ruta logística y opacidad estructural.
- El score de riesgo permitió priorizar tiendas no solo por una señal aislada, sino por la convergencia de múltiples capas de vulnerabilidad.
- La versión final de tiers generó una estructura operativamente accionable:
  - un Tier 1 pequeño y quirúrgico para intervención inmediata
  - un Tier 2 ampliado para prevención reforzada
  - un Tier 3 monitoreable
  - y un baseline creíble para el resto de la red
- El Tier 1 concentra las tiendas donde la pérdida material, el horario atípico y la exposición operativa convergen con mayor fuerza.

### Interpretación metodológica
La matriz no sustituye una investigación formal, pero sí resuelve el problema más importante de priorización:
transformar múltiples hallazgos dispersos en una secuencia clara de acción ejecutiva.

### Decisión analítica
La compañía ya no necesita mirar toda la red con la misma intensidad.  
Ahora puede intervenir primero donde el riesgo es simultáneamente más severo, más visible y más económicamente relevante.

## Próximo paso recomendado

Con la matriz de riesgo consolidada, la siguiente etapa debe enfocarse en:

1. **Traducir esta priorización a una narrativa ejecutiva visual**
2. **Preparar visualizaciones premium para comité**
3. **Construir una presentación HTML interactiva con enfoque CEO / CFO / Legal**
4. **Definir recomendaciones diferenciadas por Tier de riesgo**

La fase analítica ya permitió responder la pregunta de priorización.  
La siguiente fase debe responder la pregunta de comunicación:
**cómo convertir evidencia compleja en decisión ejecutiva clara.**